1. **_DATA INGESTION_**
Get to know the data

In [0]:
import pandas as pd
import numpy as np

- How many rows and columns does each file have? What does one row represent?
- Which columns are numbers, which are text and which are dates?
- Write down one question each file could help you answer.

In [0]:
Orders=spark.table("python.default.`1790098714770_orders`")
Orders=Orders.toPandas()

display(Orders)

OrderID,CustomerID,OrderDate,ProductID,Quantity,Discount,PaymentMethod,Status
500001,103695,2025-08-28,2003,4.0,0.1,Gateway,Completed
500002,107935,2024-05-31,2014,1.0,0.1,Wallet,Completed
500003,105141,2025-08-10,2002,1.0,0.2,CardToCard,Completed
500004,108780,2024-10-11,2012,1.0,0.1,Gateway,Completed
500005,101187,2024-02-19,2008,2.0,0.0,Gateway,Completed
500006,105545,2024-07-16,2011,4.0,0.0,Gateway,Returned
500007,106418,2025-10-15,2017,1.0,0.05,Wallet,Completed
500008,108755,2024-12-03,2014,1.0,0.0,Gateway,Completed
500009,105133,2025-04-15,2004,1.0,0.1,Gateway,Completed
500010,103111,2024-04-28,2014,2.0,0.05,Gateway,Cancelled


2.**_Clean the data (ORDERS EDA)_**
- Missing values: which columns have blanks? Should you remove, fill or keep those rows?
- Duplicates: are any orders recorded more than once?
- Values that make no sense: look at the smallest and largest numbers in each numeric column. Do the productprices look sensible for each item? .
- Inconsistent text: are the same cities spelled in more than one way?
- Broken links: does every order point to a customer and product that actually exist?

In [0]:
Orders.dtypes

OrderID            int64
CustomerID         int64
OrderDate         object
ProductID          int64
Quantity         float64
Discount         float64
PaymentMethod     object
Status            object
dtype: object

In [0]:
Orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50120 entries, 0 to 50119
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   OrderID        50120 non-null  int64  
 1   CustomerID     50120 non-null  int64  
 2   OrderDate      50085 non-null  object 
 3   ProductID      50120 non-null  int64  
 4   Quantity       50040 non-null  float64
 5   Discount       49899 non-null  float64
 6   PaymentMethod  49668 non-null  object 
 7   Status         50120 non-null  object 
dtypes: float64(2), int64(3), object(3)
memory usage: 3.1+ MB


In [0]:
# Check how many missing values are in each column
print(Orders.isna().sum())

OrderID            0
CustomerID         0
OrderDate         35
ProductID          0
Quantity          80
Discount         221
PaymentMethod    452
Status             0
dtype: int64


In [0]:
# Show rows that contain at least one missing value
display(Orders[Orders.isna().any(axis=1)])

OrderID,CustomerID,OrderDate,ProductID,Quantity,Discount,PaymentMethod,Status
500045,100241,2025-06-18,2017,2.0,0.05,null,Completed
500114,107642,2025-11-30,2007,3.0,0.0,null,Completed
500116,102851,2026-06-04,2001,1.0,0.05,null,Completed
500120,104593,2026-01-19,2001,5.0,0.0,null,Completed
500134,100871,2025-02-02,2017,1.0,null,CardToCard,Completed
500174,106216,2025-03-15,2016,1.0,0.1,null,Completed
500199,105852,2025-04-07,2011,2.0,0.0,null,Completed
500247,102218,2025-07-14,2005,3.0,null,Cash,Completed
500283,105484,2025-01-27,2004,2.0,0.1,null,Completed
500305,102046,2025-08-02,2007,2.0,0.2,null,Completed


In [0]:
#Checking order duplicates
Orders.duplicated().sum()

np.int64(120)

In [0]:
# Checking for duplicate OrderIDs
print(Orders["OrderID"].duplicated().sum())

120


In [0]:
# Check the smallest and largest Quantity
print("Minimum Quantity:", Orders["Quantity"].min())
print("Maximum Quantity:", Orders["Quantity"].max())

# Check the smallest and largest Discount
print("Minimum Discount:", Orders["Discount"].min())
print("Maximum Discount:", Orders["Discount"].max())

Minimum Quantity: -2.0
Maximum Quantity: 5.0
Minimum Discount: 0.0
Maximum Discount: 0.3


In [0]:
 # Check different payment methods
print(Orders["PaymentMethod"].unique())

# Check different order statuses
print(Orders["Status"].unique())

['Gateway' 'Wallet' 'CardToCard' 'Cash' None]
['Completed' 'Returned' 'Cancelled']


In [0]:
# Counting the number of orders for each payment method
Orders["PaymentMethod"].value_counts()

PaymentMethod
Gateway       23877
CardToCard    11981
Wallet         8804
Cash           5006
Name: count, dtype: int64

In [0]:
# Counting each order status
print(Orders["Status"].value_counts())

Status
Completed    46105
Cancelled     2477
Returned      1538
Name: count, dtype: int64


In [0]:
 # Remove spaces before and after text
Orders["PaymentMethod"] = Orders["PaymentMethod"].str.strip()
Orders["Status"] = Orders["Status"].str.strip()

print(Orders["PaymentMethod"].unique())
print(Orders["Status"].unique())

['Gateway' 'Wallet' 'CardToCard' 'Cash' None]
['Completed' 'Returned' 'Cancelled']


In [0]:
# 1. Convert transaction_date into a proper date
Orders["OrderDate"] = pd.to_datetime(Orders["OrderDate"])

# 2. Extract the year
Orders["year"] = Orders["OrderDate"].dt.year

# 3. Extract the month number
Orders["month"] = Orders["OrderDate"].dt.month

# 4. Extract the day of the month
Orders["day"] = Orders["OrderDate"].dt.day

# 5. Extract the day name
Orders["day_of_week"] = Orders["OrderDate"].dt.day_name()

# 6. Create a month-year column
Orders["month_year"] = Orders["OrderDate"].dt.to_period("M")

# 7. Display the result
display(Orders)

OrderID,CustomerID,OrderDate,ProductID,Quantity,Discount,PaymentMethod,Status,year,month,day,day_of_week,month_year
500001,103695,2025-08-28T00:00:00.000Z,2003,4.0,0.1,Gateway,Completed,2025.0,8.0,28.0,Thursday,667
500002,107935,2024-05-31T00:00:00.000Z,2014,1.0,0.1,Wallet,Completed,2024.0,5.0,31.0,Friday,652
500003,105141,2025-08-10T00:00:00.000Z,2002,1.0,0.2,CardToCard,Completed,2025.0,8.0,10.0,Sunday,667
500004,108780,2024-10-11T00:00:00.000Z,2012,1.0,0.1,Gateway,Completed,2024.0,10.0,11.0,Friday,657
500005,101187,2024-02-19T00:00:00.000Z,2008,2.0,0.0,Gateway,Completed,2024.0,2.0,19.0,Monday,649
500006,105545,2024-07-16T00:00:00.000Z,2011,4.0,0.0,Gateway,Returned,2024.0,7.0,16.0,Tuesday,654
500007,106418,2025-10-15T00:00:00.000Z,2017,1.0,0.05,Wallet,Completed,2025.0,10.0,15.0,Wednesday,669
500008,108755,2024-12-03T00:00:00.000Z,2014,1.0,0.0,Gateway,Completed,2024.0,12.0,3.0,Tuesday,659
500009,105133,2025-04-15T00:00:00.000Z,2004,1.0,0.1,Gateway,Completed,2025.0,4.0,15.0,Tuesday,663
500010,103111,2024-04-28T00:00:00.000Z,2014,2.0,0.05,Gateway,Cancelled,2024.0,4.0,28.0,Sunday,651


In [0]:
# Check the date range of the orders
print("First order:", Orders["OrderDate"].min())
print("Last order:", Orders["OrderDate"].max())

First order: 2024-01-01 00:00:00
Last order: 2026-06-30 00:00:00
